# MIE 402 · Lab 2 saved-data analysis

**For students who have already completed the experiment.** You do **not** need to record the pendulum again. This notebook uses your saved `ExpData.mat` file (or a ZIP containing that MAT file). All code is supplied; run the cells from top to bottom. Your work is to inspect the result and explain it in your own report.

**What to submit in your Lab 2 report:**
1. The `Lab2_report_figure.png` produced below: **black dots are measured positions converted to angle; the green dashed curve is a fitted model**, including where no position was recorded. Do not call the dashed curve measured data.
2. Your fitted frequency and period, with units, and a brief comparison with your theoretical pendulum prediction. State that the fit is approximate if tracking is incomplete.
3. The number and percentage of valid tracked frames and the longest gap, shown in the results. Explain that missing frames limit confidence. Keep the original MAT/video as raw evidence.

The notebook does not invent missing measurements. It uses only valid recorded points for fitting. If you still have the raw video, you may reprocess it in the analyzer; no new physical experiment is required. If you only have the saved MAT file, use it with the stated limitations.


## 1 · Upload the saved data

Click the play button, then choose your `ExpData.mat` or a ZIP containing **one** MAT file. If Colab asks to connect to a runtime, accept it. Do not upload only the MP4: this notebook needs the analyzer's saved coordinates.


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
candidates = [Path(name) for name in uploaded if Path(name).suffix.lower() in ('.mat', '.zip')]
if len(candidates) != 1:
    raise ValueError('Upload exactly one ExpData .mat file or one ZIP containing it.')
source_path = candidates[0]
print('Selected:', source_path.name)


## 2 · Load positions and fit the oscillation

The analyzer saves time and pivot-relative bob position in millimeters, with vertical position positive upward. We calculate angle from the hanging-down direction using `atan2(x, -y)`. The fit allows the oscillation amplitude to decrease over time. Its smooth line is a model, **not a reconstruction of untracked frames**. This step may take about a minute.


In [ ]:
import json
import struct
import zipfile
import zlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def read_expdata(path: Path) -> np.ndarray:
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path) as z:
            files = [n for n in z.namelist() if n.lower().endswith(".mat")]
            if len(files) != 1:
                raise ValueError("ZIP must contain exactly one MAT file.")
            raw = z.read(files[0])
    else:
        raw = path.read_bytes()
    if not raw.startswith(b"MATLAB 5.0 MAT-file") or raw[126:128] != b"IM":
        raise ValueError("Expected a little-endian MATLAB v5 MAT file.")
    kind, size = struct.unpack_from("<II", raw, 128)
    if kind == 15:  # miCOMPRESSED
        block = zlib.decompress(raw[136:136 + size])
    elif kind == 14:  # miMATRIX
        block = raw[128:136 + size]
    else:
        raise ValueError(f"Unsupported MAT element type {kind}.")
    kind, size = struct.unpack_from("<II", block, 0)
    if kind != 14:
        raise ValueError("Expected a MATLAB matrix element.")
    off = 8
    elements = []
    while off < 8 + size:
        element_type, nbytes = struct.unpack_from("<II", block, off)
        off += 8
        elements.append((element_type, block[off:off + nbytes]))
        off += (nbytes + 7) // 8 * 8
    name = elements[2][1].decode("ascii")
    if name != "ExpData" or elements[3][0] != 9:
        raise ValueError("MAT file must contain a double-precision ExpData matrix.")
    dims = tuple(np.frombuffer(elements[1][1], dtype="<i4"))
    data = np.frombuffer(elements[3][1], dtype="<f8").reshape(dims, order="F").copy()
    if data.ndim != 2 or data.shape[1] < 3:
        raise ValueError("ExpData needs time, bob x, and bob y columns.")
    return data


def linear_sine(t: np.ndarray, y: np.ndarray, freq: float, decay: float = 0,
                trend: bool = False):
    u = t - t[0]
    envelope = np.exp(-decay * u)
    columns = [np.ones(len(t))]
    if trend:
        columns.append(u)
    columns += [envelope * np.sin(2 * np.pi * freq * u),
                envelope * np.cos(2 * np.pi * freq * u)]
    matrix = np.column_stack(columns)
    coef = np.linalg.lstsq(matrix, y, rcond=None)[0]
    fitted = matrix @ coef
    return np.mean((y - fitted) ** 2), coef, fitted


def search_constant_sine(t, y, fmin, fmax):
    grid = np.linspace(fmin, fmax, 2500)
    errors = np.array([linear_sine(t, y, f)[0] for f in grid])
    idx = int(np.argmin(errors))
    lo, hi = grid[max(0, idx - 2)], grid[min(len(grid) - 1, idx + 2)]
    for _ in range(35):
        a = lo + (hi - lo) / 3
        b = hi - (hi - lo) / 3
        if linear_sine(t, y, a)[0] < linear_sine(t, y, b)[0]:
            hi = b
        else:
            lo = a
    f = (lo + hi) / 2
    mse, coef, fitted = linear_sine(t, y, f)
    return f, mse, coef, fitted


def search_damped_sine(t, y, center, fmin, fmax):
    best = (float("inf"), None, None, None, None)
    low, high = max(fmin, center - .20), min(fmax, center + .20)
    for decay in np.linspace(0, .4, 81):
        for freq in np.linspace(low, high, 121):
            mse, coef, fitted = linear_sine(t, y, freq, decay, trend=True)
            if mse < best[0]:
                best = mse, freq, decay, coef, fitted
    mse, freq, decay, coef, fitted = best
    for span in (.015, .003, .0006):
        for d in np.linspace(max(0, decay - span), decay + span, 31):
            for f in np.linspace(max(fmin, freq - span), min(fmax, freq + span), 31):
                candidate = linear_sine(t, y, f, d, trend=True)
                if candidate[0] < mse:
                    mse, freq, decay, coef, fitted = candidate[0], f, d, candidate[1], candidate[2]
    return freq, decay, mse, coef, fitted


def evaluate_model(t, reference_t, coef, freq, decay=0, trend=False):
    u = t - reference_t
    base = coef[0] + (coef[1] * u if trend else 0)
    a, b = coef[-2:]
    return base + np.exp(-decay * u) * (a * np.sin(2 * np.pi * freq * u)
                                          + b * np.cos(2 * np.pi * freq * u))




In [ ]:
data = read_expdata(source_path)
times = np.asarray(data[:, 0], dtype=float)
if not np.all(np.isfinite(times)) or np.any(np.diff(times) <= 0):
    raise ValueError('Time values must be finite and strictly increasing.')
valid = np.isfinite(data[:, 1]) & np.isfinite(data[:, 2])
if valid.sum() < 30:
    raise ValueError('Fewer than 30 valid tracked positions: there is not enough information for this fit. Contact the instructor with your original files.')
t = times[valid]
theta = np.unwrap(np.arctan2(data[valid, 1], -data[valid, 2]))
missing = 1 - valid.mean()
gap_edges = np.diff(np.r_[False, ~valid, False].astype(int))
starts = np.flatnonzero(gap_edges == 1)
ends = np.flatnonzero(gap_edges == -1) - 1
longest_gap = (int(np.max(ends - starts + 1)) * float(np.median(np.diff(times)))) if len(starts) else 0.0

constant = search_constant_sine(t, theta, 0.1, 5.0)
damped = search_damped_sine(t, theta, constant[0], 0.1, 5.0)
f_fit, decay, mse, coefficients, fitted_at_points = damped
period = 1 / f_fit
rmse_deg = float(np.degrees(np.sqrt(mse)))

print(f'Valid tracked frames: {valid.sum()} / {len(valid)} ({100*valid.mean():.1f}%)')
print(f'Missing frames: {100*missing:.1f}% | Longest gap: {longest_gap:.2f} s')
print(f'Approximate fitted frequency: {f_fit:.3f} Hz | Period: {period:.3f} s')
print(f'Fit RMSE on measured points: {rmse_deg:.2f} degrees')
if missing > 0.10 or longest_gap > 0.05:
    print('CAUTION: Tracking is incomplete. Use the figure and frequency with an explicit limitation statement; do not present the model as recovered data.')


## 3 · Make the figure for your report

Insert the PNG into your report. The dots are the observations; the dashed curve is the damped-sine model. The bottom strip makes gaps in tracking visible. If a visible portion of the dashed line misses the measured dots, say so rather than claiming an excellent fit. The figure also exports as a PDF.


In [ ]:
dense = np.linspace(t[0], t[-1], 2500)
model = evaluate_model(dense, t[0], coefficients, f_fit, decay, trend=True)
fig, (ax, availability) = plt.subplots(2, 1, figsize=(11, 6.7),
                                      gridspec_kw={'height_ratios': [4, .45]},
                                      sharex=True, constrained_layout=True)
ax.plot(dense, np.degrees(model), color='#087e64', lw=1.7, ls='--',
        label=f'Damped-sine model: f = {f_fit:.3f} Hz (not measured in gaps)')
ax.scatter(t, np.degrees(theta), s=11, color='#17243c', zorder=3,
           label=f'Measured angle: {valid.sum()} of {len(valid)} frames')
ax.set(ylabel='Angle from downward vertical (degrees)',
       title='Single pendulum: observed angle and fitted model')
ax.grid(alpha=.25)
ax.legend(loc='best', fontsize=9)
ax.text(.01, .02, f'Valid frames: {100*valid.mean():.1f}%  |  Longest missing interval: {longest_gap:.2f} s',
        transform=ax.transAxes, fontsize=9, va='bottom',
        bbox={'facecolor':'white', 'alpha':.88, 'edgecolor':'#cbd5e1'})
availability.scatter(times[valid], np.zeros(valid.sum()), s=2, color='#17243c')
availability.set(xlabel='Time (s)', ylabel='Valid\nframes', yticks=[], ylim=(-.3, .3))
availability.spines[['left', 'right', 'top']].set_visible(False)
fig.savefig('Lab2_report_figure.png', dpi=220, bbox_inches='tight')
fig.savefig('Lab2_report_figure.pdf', bbox_inches='tight')
plt.show()


## 4 · Download and write your interpretation

Run the next cell to download one ZIP containing the figure and a short results text file. You may put the PNG in Word or use the PDF in another report tool. **Do not submit the ZIP as your whole lab report.** Write the analysis in your own words.

Suggested caption to adapt (replace the bracketed values with your output):

> **Figure X.** Measured single-pendulum angle (dots) and a damped-sine fit (dashed line). The fit gives an approximate frequency of **[f] Hz** and period of **[T] s**. **[n] of [N]** frames contained valid tracked positions; the longest missing interval was **[gap] s**. The dashed line across gaps is a model prediction, not a measurement.

In your discussion, compare the fitted period with your theoretical prediction. If many frames are missing, say that the frequency estimate is provisional and avoid claiming that the model validates the entire trajectory. Do not report the fitted decay coefficient as a measured air-drag coefficient: pivot friction and tracking errors also affect the trace.


In [ ]:
from google.colab import files
from zipfile import ZipFile, ZIP_DEFLATED

results_text = f'''MIE 402 Lab 2 saved-data analysis
Source file: {source_path.name}
Valid tracked frames: {valid.sum()} of {len(valid)} ({100*valid.mean():.1f}%)
Missing frames: {100*missing:.1f}%
Longest missing interval: {longest_gap:.2f} s
Approximate fitted frequency: {f_fit:.3f} Hz
Approximate fitted period: {period:.3f} s
Fit RMSE at measured points: {rmse_deg:.2f} degrees

Black dots in the figure are measured tracked positions. The dashed line is
a model and is not measured data where tracking is missing. Interpret the
estimate cautiously if many positions are absent.
'''
Path('Lab2_results.txt').write_text(results_text, encoding='utf-8')
with ZipFile('Lab2_report_package.zip', 'w', ZIP_DEFLATED) as package:
    for name in ('Lab2_report_figure.png', 'Lab2_report_figure.pdf', 'Lab2_results.txt'):
        package.write(name)
files.download('Lab2_report_package.zip')
